In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss,accuracy_score,balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Wisconsin/")

In [2]:
df = pd.read_csv("BreastCancer.csv",index_col="Code")
df

,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
Code,,,,,,,,,,
61634,5,4,3,1,2,2,2,3,1,Benign
63375,9,1,2,6,4,10,7,7,2,Malignant
76389,10,4,7,2,2,8,6,1,1,Malignant
95719,6,10,10,10,8,10,7,10,7,Malignant
128059,1,1,1,1,2,5,5,1,1,Benign
...,...,...,...,...,...,...,...,...,...,...
1369821,10,10,10,10,5,10,10,10,7,Malignant
1371026,5,10,10,10,4,10,5,6,3,Malignant
1371920,5,1,1,1,2,1,3,2,1,Benign


In [3]:
le = LabelEncoder()

In [4]:
df['Class'] = le.fit_transform(df['Class'])

In [5]:
X,y = df.drop('Class',axis=1), df['Class']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [13]:
dtc = DecisionTreeClassifier(random_state=26,max_depth=1)

In [8]:
lr = LogisticRegression(random_state=26)

In [9]:
nb = GaussianNB()

In [18]:
knn = KNeighborsClassifier(n_neighbors=3,weights="SAMME")

In [20]:
estimators = [dtc,lr,nb]
no_of_estimators = np.arange(1,100)
scores=[]
for e in tqdm(estimators):
    for n in no_of_estimators: 
        ada = AdaBoostClassifier(estimator=e,n_estimators=n,random_state=26)
        ada.fit(X_train,y_train)
        y_pred = ada.predict(X_test)
        y_pred_prob = ada.predict_proba(X_test)
        scores.append([e,n,log_loss(y_test,y_pred_prob),accuracy_score(y_test,y_pred),balanced_accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=["Estimator","Number of Estimators","Log Loss","Accuracy Score","Balanced Accuracy Score"])
df_scores.sort_values("Log Loss")

100%|█████████████████████████████████████████████| 3/3 [00:51<00:00, 17.16s/it]


,Estimator,Number of Estimators,Log Loss,Accuracy Score,Balanced Accuracy Score
268,GaussianNB(),71,0.108219,0.961905,0.967693
281,GaussianNB(),84,0.108400,0.961905,0.967693
276,GaussianNB(),79,0.108619,0.966667,0.971316
280,GaussianNB(),83,0.110354,0.966667,0.971316
270,GaussianNB(),73,0.110412,0.957143,0.960749
...,...,...,...,...,...
197,LogisticRegression(),99,0.663401,0.957143,0.950785
208,GaussianNB(),11,0.835368,0.709524,0.596316
198,GaussianNB(),1,1.170585,0.947619,0.950181
204,GaussianNB(),7,1.247570,0.585714,0.684783
